# Uncertainty Analysis — Monte Carlo Simulation

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

np.random.seed(42)

## Lognormal Sampling

In [ ]:
# Simulate CO2e for a product with known median and uncertainty range
# Using lognormal distribution (common for emission factors)
median_co2e = 70  # kg CO2e (e.g., smartphone)
gsd = 1.5  # geometric standard deviation

mu = np.log(median_co2e)
sigma = np.log(gsd)

n_samples = 10000
mc_samples = np.random.lognormal(mean=mu, sigma=sigma, size=n_samples)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(mc_samples, bins=80, color='#2ecc71', edgecolor='white', density=True)
x_pdf = np.linspace(mc_samples.min(), np.percentile(mc_samples, 99), 200)
pdf = stats.lognorm.pdf(x_pdf, s=sigma, scale=median_co2e)
axes[0].plot(x_pdf, pdf, 'r-', linewidth=2, label='Lognormal PDF')
axes[0].axvline(median_co2e, color='#e74c3c', linestyle='--', label=f'Median = {median_co2e}')
axes[0].set_xlabel('CO\u2082e (kg)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Monte Carlo Samples (n={n_samples:,})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(mc_samples, bins=80, color='#3498db', edgecolor='white', density=True, cumulative=True)
axes[1].axhline(0.05, color='gray', linestyle=':', alpha=0.5)
axes[1].axhline(0.95, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('CO\u2082e (kg)')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('Cumulative Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/monte_carlo_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

p5, p50, p95 = np.percentile(mc_samples, [5, 50, 95])
print(f'5th percentile:  {p5:.1f} kg CO\u2082e')
print(f'50th percentile: {p50:.1f} kg CO\u2082e')
print(f'95th percentile: {p95:.1f} kg CO\u2082e')
print(f'Mean:            {mc_samples.mean():.1f} kg CO\u2082e')
print(f'Std:             {mc_samples.std():.1f} kg CO\u2082e')

## Convergence Analysis

In [ ]:
iteration_counts = np.logspace(1, 4, 50).astype(int)
means = []
ci_lower = []
ci_upper = []

all_samples = np.random.lognormal(mean=mu, sigma=sigma, size=10000)

for n in iteration_counts:
    subset = all_samples[:n]
    m = subset.mean()
    se = subset.std() / np.sqrt(n)
    means.append(m)
    ci_lower.append(m - 1.96 * se)
    ci_upper.append(m + 1.96 * se)

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(iteration_counts, ci_lower, ci_upper, alpha=0.3, color='#3498db', label='95% CI')
ax.plot(iteration_counts, means, '-', color='#2c3e50', linewidth=1.5, label='Running mean')
ax.axhline(all_samples.mean(), color='#e74c3c', linestyle='--', alpha=0.7, label='True mean')
ax.set_xscale('log')
ax.set_xlabel('Number of Iterations')
ax.set_ylabel('Mean CO\u2082e (kg)')
ax.set_title('Monte Carlo Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/mc_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## Bootstrap Confidence Intervals

In [ ]:
n_bootstrap = 5000
bootstrap_means = []
observed = mc_samples[:200]  # pretend we have 200 observations

for _ in range(n_bootstrap):
    resample = np.random.choice(observed, size=len(observed), replace=True)
    bootstrap_means.append(resample.mean())

bootstrap_means = np.array(bootstrap_means)
ci_lo, ci_hi = np.percentile(bootstrap_means, [2.5, 97.5])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(bootstrap_means, bins=60, color='#9b59b6', edgecolor='white', density=True)
ax.axvline(ci_lo, color='#e74c3c', linestyle='--', linewidth=2, label=f'2.5% = {ci_lo:.1f}')
ax.axvline(ci_hi, color='#e74c3c', linestyle='--', linewidth=2, label=f'97.5% = {ci_hi:.1f}')
ax.axvline(observed.mean(), color='#2ecc71', linestyle='-', linewidth=2, label=f'Observed mean = {observed.mean():.1f}')
ax.set_xlabel('Mean CO\u2082e (kg)')
ax.set_ylabel('Density')
ax.set_title(f'Bootstrap Distribution of Mean (n_boot={n_bootstrap:,})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/bootstrap_ci.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Bootstrap 95% CI: [{ci_lo:.1f}, {ci_hi:.1f}] kg CO\u2082e')

## Sensitivity Analysis

In [ ]:
# One-at-a-time (OAT) perturbation analysis
# Simulate a product footprint = materials + transport + energy + packaging
base_params = {
    'materials_co2e': 35.0,
    'transport_co2e': 12.0,
    'energy_co2e': 15.0,
    'packaging_co2e': 5.0,
    'eol_co2e': 3.0,
}

base_total = sum(base_params.values())
perturbation = 0.20  # +/- 20%

deltas_low = {}
deltas_high = {}

for param, val in base_params.items():
    # Perturb one parameter at a time
    low_total = base_total - val + val * (1 - perturbation)
    high_total = base_total - val + val * (1 + perturbation)
    deltas_low[param] = low_total - base_total
    deltas_high[param] = high_total - base_total

# Sort by absolute impact
sorted_params = sorted(base_params.keys(), key=lambda p: abs(deltas_high[p]))

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(sorted_params))

for i, param in enumerate(sorted_params):
    ax.barh(i, deltas_high[param], color='#e74c3c', height=0.4, label='+20%' if i == 0 else '')
    ax.barh(i, deltas_low[param], color='#3498db', height=0.4, label='-20%' if i == 0 else '')

ax.set_yticks(y_pos)
ax.set_yticklabels([p.replace('_co2e', '') for p in sorted_params])
ax.set_xlabel('\u0394 CO\u2082e (kg) from baseline')
ax.set_title(f'Tornado Diagram — OAT Sensitivity (\u00b1{perturbation:.0%})')
ax.axvline(0, color='black', linewidth=0.5)
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../figures/tornado_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Baseline total: {base_total:.1f} kg CO\u2082e')
for p in reversed(sorted_params):
    print(f'  {p}: {deltas_low[p]:+.1f} / {deltas_high[p]:+.1f} kg')